# Huấn luyện XGBoost rủi ro trên Kaggle (GPU T4 x2)

Notebook này chuyển đổi `step_train_risk_model.py` sang dạng chạy trên Kaggle:
- Đọc `risk_dataset.csv` (21 đặc trưng + nhãn rủi ro) từ Kaggle Input (tự động tìm file).
- Cân bằng nhãn (undersample lớp âm, neg:pos = 5:1).
- Chuẩn hóa causal per-sequence z-score (Welford, min_n=50) — giống step5 suy luận.
- Đánh giá GroupKFold theo sequence (không rò rỉ) + temporal hold-out.
- Tự động phát hiện GPU T4; nếu có dùng `tree_method='gpu_hist'`, `predictor='gpu_predictor'`; ngược lại dùng CPU.
- Cuối cùng: nén toàn bộ đầu ra thành 1 ZIP để tải về máy.

Trước khi chạy: up `risk_dataset.csv` vào Data → Add data → Upload trên Kaggle. File đầu ra lưu vào `/kaggle/working/`.

In [ ]:
# ── Cell 1 · Bootstrap: cài gói, import, phát hiện GPU/Kaggle ──────────────────────
import sys, subprocess, os, json, shutil
from pathlib import Path
import importlib.util

WORK = Path('/kaggle/working')
INPUT = Path('/kaggle/input')
WORK.mkdir(parents=True, exist_ok=True)

def ensure(pkg, import_name=None):
    if importlib.util.find_spec(import_name or pkg) is None:
        print(f'Cài {pkg}...')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

ensure('numpy', 'numpy')
ensure('pandas', 'pandas')
ensure('scikit-learn', 'sklearn')
ensure('xgboost', 'xgboost')

import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.model_selection import GroupKFold

print('numpy', np.__version__)
print('pandas', pd.__version__)
print('xgboost', xgb.__version__)

def xgb_gpu_available():
    try:
        import numpy as _np
        X = _np.random.rand(10, 3).astype(_np.float32)
        y = _np.random.randint(0, 2, 10)
        dtrain = xgb.DMatrix(X, label=y)
        params = {'tree_method': 'gpu_hist', 'predictor': 'gpu_predictor'}
        xgb.train(params, dtrain, num_boost_round=1)
        return True
    except Exception as e:
        print('Không dùng được GPU XGBoost:', str(e)[:120])
        return False

USE_GPU = xgb_gpu_available()
print('XGBoost thiết bị:', 'GPU (T4)' if USE_GPU else 'CPU')

In [ ]:
# ── Cell 2 · CONFIG: đường dẫn và siêu tham số ─────────────────────────────────────
from pathlib import Path

# ================= TỰ ĐỘNG TÌM risk_dataset.csv TRÊN KAGGLE =========================
WORK = Path('/kaggle/working')
INPUT = Path('/kaggle/input')

RISK_CSV = None
for search_root in [INPUT, WORK]:
    for p in search_root.rglob('risk_dataset.csv'):
        RISK_CSV = p
        print(f'Tìm thấy risk_dataset.csv tại: {p}')
        break
    if RISK_CSV is not None:
        break

if RISK_CSV is None:
    raise FileNotFoundError(
        'Không tìm thấy risk_dataset.csv trên Kaggle. '
        'Hãy up file vào Data → Add data → Upload.'
    )

ARTIFACT_INPUT = RISK_CSV.parent

# ================= OUTPUT (read-write, lưu tại /kaggle/working) ====================
ARTIFACT = WORK / 'artifacts'
MODELS_DIR = WORK / 'models'
RESULTS_DIR = WORK / 'results'
for d in (ARTIFACT, MODELS_DIR, RESULTS_DIR):
    d.mkdir(parents=True, exist_ok=True)

# ================= ĐƯỜNG DẪN FILE ĐẦU RA ===========================================
BALANCED_CSV = ARTIFACT / 'risk_dataset_balanced.csv'
MODEL_PATH = MODELS_DIR / 'xgboost_risk.json'
BASELINE_PATH = MODELS_DIR / 'xgboost_risk_baseline.json'
SUMMARY_PATH = RESULTS_DIR / 'risk_model_training_summary.json'

# ================= SIÊU THAM SỐ ====================================================
NEG_POS_RATIO = 5.0
HOLDOUT_FRACTION = 0.2
MIN_NORM_SAMPLES = 50
N_FOLDS = 5
SEED = 42

FEATURES = [
    'conf', 'conf_std5', 'conf_delta', 'track_age', 'hit_ratio',
    'time_since_update', 'track_state', 'min_side_px', 'aspect_dev',
    'dist_edge', 'is_trunc', 'speed_norm', 'pred_uncert', 'lap_var_log',
    'brightness', 'motion_blur_ratio', 'nearest_dist_norm',
    'n_neighbors_r2_norm', 'overlap_max', 'frame_det_delta',
    'n_det_expected_ratio',
]

XGB_PARAMS_CPU = dict(
    n_estimators=300,
    max_depth=2,
    learning_rate=0.03,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=5,
    reg_lambda=5.0,
    reg_alpha=0.5,
    objective='binary:logistic',
    eval_metric='aucpr',
    tree_method='hist',
    early_stopping_rounds=30,
    n_jobs=4,
    random_state=SEED,
)

XGB_PARAMS_GPU = dict(
    n_estimators=300,
    max_depth=2,
    learning_rate=0.03,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=5,
    reg_lambda=5.0,
    reg_alpha=0.5,
    objective='binary:logistic',
    eval_metric='aucpr',
    tree_method='gpu_hist',
    predictor='gpu_predictor',
    early_stopping_rounds=30,
    random_state=SEED,
)

XGB_PARAMS = XGB_PARAMS_GPU if USE_GPU else XGB_PARAMS_CPU
print('Sử dụng thiết bị:', 'GPU (T4 x2)' if USE_GPU else 'CPU')

In [ ]:
# ── Cell 3 · Đọc dữ liệu có nhãn ───────────────────────────────────────────────────
print(f'Đọc dữ liệu: {RISK_CSV}')
print('Tồn tại:', RISK_CSV.exists())
df = pd.read_csv(RISK_CSV)
labeled = df[df['label'].notna()].copy()
labeled['label'] = labeled['label'].astype(int)
labeled = labeled.reset_index(drop=True)

missing = [f for f in FEATURES if f not in labeled.columns]
if missing:
    raise ValueError(f'Thiếu đặc trưng: {missing}')

print(
    f'Dữ liệu gốc có nhãn: {len(labeled)} mẫu | '
    f'dương: {int(labeled.label.sum())} ({labeled.label.mean():.2%}) | '
    f'sequence: {labeled.sequence.nunique()}'
)

In [ ]:
# ── Cell 4 · Cân bằng nhãn (undersample lớp âm, neg:pos = 5:1) ─────────────────────
def balance_labels(df, ratio, seed):
    pos = df[df.label == 1]
    neg = df[df.label == 0]
    n_neg = min(len(neg), int(round(len(pos) * ratio)))
    neg_sampled = neg.sample(n=n_neg, random_state=seed)
    out = pd.concat([pos, neg_sampled], ignore_index=False)
    return out.sample(frac=1.0, random_state=seed)

balanced = balance_labels(labeled, NEG_POS_RATIO, SEED)
balanced.to_csv(BALANCED_CSV, index=False, encoding='utf-8-sig')
print(
    f'Đã lưu tập cân bằng: {BALANCED_CSV} | {len(balanced)} mẫu | '
    f'dương: {int(balanced.label.sum())} ({balanced.label.mean():.2%})'
)

In [ ]:
# ── Cell 5 · Chuẩn hóa causal per-sequence z-score (Welford, min_n=50) ─────────────
def add_causal_sequence_zscore(df, min_n=MIN_NORM_SAMPLES):
    '''Chỉ dùng các mẫu quá khứ của cùng sequence để tính mean/var (giống step5).'''
    X = df[FEATURES].to_numpy(np.float64)
    out = np.empty_like(X)
    seq = df['sequence'].to_numpy()
    order = np.lexsort((df['track_id'].to_numpy(), df['window_start'].to_numpy(), seq))
    start, n = 0, len(order)
    while start < n:
        cur = seq[order[start]]
        end = start
        while end < n and seq[order[end]] == cur:
            end += 1
        mean = np.zeros(X.shape[1])
        m2 = np.zeros(X.shape[1])
        cnt = 0
        for k in range(start, end):
            i = order[k]
            x = X[i]
            cnt += 1
            if cnt <= min_n:
                out[i] = x
            else:
                var = m2 / (cnt - 1)
                sd = np.sqrt(np.maximum(var, 1e-6))
                out[i] = (x - mean) / sd
            d = x - mean
            mean = mean + d / cnt
            m2 = m2 + d * (x - mean)
        start = end
    return out.astype(np.float32)

print(f'Chuẩn hóa causal z-score theo sequence (min_n={MIN_NORM_SAMPLES})...')
X_norm = add_causal_sequence_zscore(labeled, MIN_NORM_SAMPLES)
balanced = balanced.copy()
balanced[FEATURES] = X_norm[balanced.index.to_numpy()]

X = balanced[FEATURES].to_numpy(np.float32)
y = balanced['label'].to_numpy(int)
groups = balanced['sequence'].to_numpy()

# Dữ liệu gốc (không cân bằng) để đánh giá OOF phản ánh thực tế
X_orig = X_norm
y_orig = labeled['label'].to_numpy(int)
groups_orig = labeled['sequence'].to_numpy()

pos_weight = float((1 - y.mean()) / max(y.mean(), 1e-6))
params = dict(XGB_PARAMS)
params['scale_pos_weight'] = pos_weight
print(f'scale_pos_weight (sau cân bằng): {pos_weight:.3f}')

In [ ]:
# ── Cell 6 · GroupKFold theo sequence (đánh giá trên phân phối nhãn gốc) ───────────
def evaluate_group_cv(X, y, groups, params, ratio):
    n_groups = len(np.unique(groups))
    n_splits = max(2, min(N_FOLDS, n_groups))
    oof = np.full(len(y), np.nan)
    rows = []
    rng = np.random.RandomState(SEED)
    for fold, (tr, va) in enumerate(GroupKFold(n_splits=n_splits).split(X, y, groups)):
        pos = np.where(y[tr] == 1)[0]
        neg = np.where(y[tr] == 0)[0]
        n_neg = min(len(neg), int(round(len(pos) * ratio)))
        neg = rng.choice(neg, size=n_neg, replace=False)
        idx = np.concatenate([pos, neg])
        model = xgb.XGBClassifier(**{**params, 'scale_pos_weight': ratio})
        model.fit(X[tr][idx], y[tr][idx], eval_set=[(X[va], y[va])], verbose=False)
        oof[va] = model.predict_proba(X[va])[:, 1]
        auc = roc_auc_score(y[va], oof[va]) if len(np.unique(y[va])) > 1 else float('nan')
        ap = average_precision_score(y[va], oof[va]) if len(np.unique(y[va])) > 1 else float('nan')
        rows.append({
            'fold': fold,
            'n_train': int(len(idx)),
            'n_val': int(len(va)),
            'auc': None if np.isnan(auc) else float(auc),
            'pr_auc': None if np.isnan(ap) else float(ap),
            'best_iteration': int(getattr(model, 'best_iteration', 0) or 0),
        })
        print(f'  fold {fold}: n_tr={len(idx)} n_va={len(va)} AUC={auc:.4f} PR-AUC={ap:.4f}')
    return oof, rows

print('GroupKFold theo sequence (đánh giá trên phân phối nhãn gốc):')
oof, folds = evaluate_group_cv(X_orig, y_orig, groups_orig, params, NEG_POS_RATIO)
oof_auc = float(roc_auc_score(y_orig, oof))
oof_pr_auc = float(average_precision_score(y_orig, oof))
print(f'  OOF AUC={oof_auc:.4f} | OOF PR-AUC={oof_pr_auc:.4f}')

In [ ]:
# ── Cell 7 · Fit lại model cuối trên toàn bộ tập cân bằng ──────────────────────────
final_params = {k: v for k, v in params.items() if k != 'early_stopping_rounds'}
final_model = xgb.XGBClassifier(**final_params)
final_model.fit(X, y, verbose=False)

full_auc = float(roc_auc_score(y_orig, final_model.predict_proba(X_orig)[:, 1]))
gap = full_auc - oof_auc
print(f'  AUC khi fit lại toàn bộ={full_auc:.4f} | chênh lệch overfit={gap:.4f}')

In [ ]:
# ── Cell 8 · Temporal hold-out: các sequence cuối làm test hoàn toàn ───────────────
seq_names = sorted(labeled['sequence'].unique())
n_hold = max(1, int(round(len(seq_names) * HOLDOUT_FRACTION)))
hold_seqs = set(seq_names[-n_hold:])
tr_df = balanced[~balanced['sequence'].isin(hold_seqs)]
te_df = labeled[labeled['sequence'].isin(hold_seqs)]
hold_auc = hold_pr_auc = None

if len(te_df) > 0 and te_df['label'].nunique() > 1 and len(tr_df) > 0:
    hold_model = xgb.XGBClassifier(**{**params, 'early_stopping_rounds': 30})
    hold_model.fit(
        tr_df[FEATURES].to_numpy(np.float32), tr_df['label'].to_numpy(int),
        eval_set=[(te_df[FEATURES].to_numpy(np.float32), te_df['label'].to_numpy(int))],
        verbose=False,
    )
    p_hold = hold_model.predict_proba(te_df[FEATURES].to_numpy(np.float32))[:, 1]
    y_hold = te_df['label'].to_numpy(int)
    hold_auc = float(roc_auc_score(y_hold, p_hold))
    hold_pr_auc = float(average_precision_score(y_hold, p_hold))
    print(
        f'  Temporal hold-out ({len(hold_seqs)} sequence, {len(te_df)} mẫu): '
        f'AUC={hold_auc:.4f} PR-AUC={hold_pr_auc:.4f}'
    )
else:
    print('  Temporal hold-out: bỏ qua (thiếu lớp hoặc thiếu mẫu).')

In [ ]:
# ── Cell 9 · Lưu model + summary JSON ──────────────────────────────────────────────
MODELS_DIR.mkdir(parents=True, exist_ok=True)
if MODEL_PATH.exists() and not BASELINE_PATH.exists():
    shutil.copy2(MODEL_PATH, BASELINE_PATH)
    print(f'Đã backup model cũ: {BASELINE_PATH}')
final_model.save_model(str(MODEL_PATH))
print(f'Đã lưu model mới: {MODEL_PATH}')

summary = {
    'source_csv': str(RISK_CSV),
    'balanced_csv': str(BALANCED_CSV),
    'n_labeled_original': int(len(labeled)),
    'positives_original': int(labeled.label.sum()),
    'positive_rate_original': float(labeled.label.mean()),
    'neg_pos_ratio': NEG_POS_RATIO,
    'normalization': f'causal per-sequence z-score (Welford, min_n={MIN_NORM_SAMPLES})',
    'evaluation': 'GroupKFold theo sequence tren phan phoi nhan goc; chi can bang tap train trong moi fold',
    'n_balanced': int(len(balanced)),
    'positive_rate_balanced': float(balanced.label.mean()),
    'n_groups': int(labeled.sequence.nunique()),
    'params': {k: v for k, v in final_params.items() if k != 'n_jobs'},
    'scale_pos_weight': pos_weight,
    'folds': folds,
    'oof_auc': oof_auc,
    'oof_pr_auc': oof_pr_auc,
    'full_fit_auc': full_auc,
    'overfit_gap': gap,
    'holdout_fraction': HOLDOUT_FRACTION,
    'holdout_sequences': sorted(hold_seqs),
    'holdout_n_samples': int(len(te_df)),
    'holdout_auc': hold_auc,
    'holdout_pr_auc': hold_pr_auc,
    'device': 'GPU (T4 x2)' if USE_GPU else 'CPU',
}
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
SUMMARY_PATH.write_text(json.dumps(summary, indent=2, ensure_ascii=False), encoding='utf-8')
print(f'Đã lưu summary: {SUMMARY_PATH}')

In [ ]:
# ── Cell 10 · Báo cáo tóm tắt ──────────────────────────────────────────────────────
import json as _json
from IPython.display import display, JSON
summary = _json.loads(SUMMARY_PATH.read_text(encoding='utf-8'))
print('=' * 60)
print('TÓM TẮT KẾT QUẢ HUẤN LUYỆN XGBOOST (Kaggle T4 x2)')
print('=' * 60)
for k, v in summary.items():
    if k != 'folds':
        print(f'  {k}: {v}')
print()
print('Chi tiết các fold:')
for f in summary.get('folds', []):
    print(f"  fold {f['fold']}: n_tr={f['n_train']} n_va={f['n_val']} "
          f"AUC={f['auc']:.4f} PR-AUC={f['pr_auc']:.4f} iter={f['best_iteration']}")
display(JSON(summary))

In [ ]:
# ── Cell 11 · Nén toàn bộ đầu ra thành ZIP để tải về ───────────────────────────────
import zipfile
from datetime import datetime

zip_name = f'risk_model_output_{datetime.now().strftime("%Y%m%d_%H%M%S")}.zip'
zip_path = WORK / zip_name

files_to_zip = [
    MODEL_PATH,
    BASELINE_PATH,
    SUMMARY_PATH,
    BALANCED_CSV,
]

files_to_zip = [f for f in files_to_zip if f.exists()]

if not files_to_zip:
    print('Không có file đầu ra để nén. Hãy chạy Cell 9 trước.')
else:
    with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
        for f in files_to_zip:
            zf.write(f, arcname=f.name)

    print(f'Đã nén {len(files_to_zip)} file thành: {zip_path}')
    print(f'Kích thước: {round(zip_path.stat().st_size / 1e6, 2)} MB')
    print()
    print('Danh sách file trong ZIP:')
    for f in files_to_zip:
        print(f'  - {f.name}')

    print()
    print('>>> HÃY TẢI FILE ZIP NÀY VỀ MÁY QUA BẢNG FILE BÊN TRÁI KAGGLE <<<')